In [ ]:
import pandas as pd

salario_minimo = pd.read_csv('salario_minimo_mensal_1940_2022.csv', sep=';')
salario_minimo_periodo = salario_minimo.loc[
    (salario_minimo.ano > 1949) &
    (salario_minimo.ano < 2019)
].copy()

edicoes_pato = pd.read_csv('Edicoes_Pato_Donald_Abril.csv', sep=';')
# edicoes_pato["preco_capa"] = edicoes_pato["preco_capa"].ffill()

In [ ]:
salario_minimo_periodo[["unidade", "valor"]] = (
    salario_minimo_periodo["salario_minimo"]
    .str.split(" ", n=1, expand=True)
)
salario_minimo_periodo

In [ ]:
salario_minimo_periodo.rename(
    columns={
        'valor': 'valor_sal',
        'unidade': 'unidade_sal'
    },
    inplace=True
)

In [ ]:
edicoes_pato[["unidade", "valor"]] = (
    edicoes_pato["preco_capa"]
    .str.split(" ", n=1, expand=True)
)

edicoes_pato.rename(
    columns={
        'valor': 'valor_gibi',
        'unidade': 'unidade_gibi'
    },
    inplace=True
)

In [ ]:
data = pd.merge(edicoes_pato, salario_minimo_periodo, on=['mes', 'ano'])
data

In [ ]:
data["valor_gibi_num"] = pd.to_numeric(
    data["valor_gibi"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)
data["valor_sal_num"] = pd.to_numeric(
    data["valor_sal"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
    errors="coerce"
)


data["%_sal_min"] = (data["valor_gibi_num"] / data["valor_sal_num"]) * 100

In [ ]:
data["data"] = pd.to_datetime(
    dict(year=data["ano"], month=data["mes"], day=1)
)

In [ ]:
data = data.sort_values("data")

In [ ]:
data.describe()

In [ ]:
import plotly.express as px

fig = px.line(
    data,
    x="data",
    y="%_sal_min",
    title="% do Salário Mínimo ao Longo do Tempo"
)

fig.update_layout(
    xaxis_title="Ano",
    yaxis_title="% do Salário Mínimo"
)

fig.show()

In [ ]:
import pandas as pd

# Remove NaN
df2 = data#.dropna(subset=['%_sal_min']).copy()

 # MÁXIMO
i_max = df2['%_sal_min'].idxmax()
max_row = df2.loc[i_max]

# MÍNIMO
i_min = df2['%_sal_min'].idxmin()
min_row = df2.loc[i_min]

# MÉDIA E MEDIANA
media = df2['%_sal_min'].mean()
mediana = df2['%_sal_min'].median()


print("MAIOR CUSTO RELATIVO:")
print(
    f"Data: {max_row['data']:%d/%m/%Y} | "
    f"Edição: {max_row['edicao_numero']} | "
    f"{max_row['%_sal_min']:.2f}% do salário mínimo "
    f"(Gibi={max_row['valor_gibi_num']:.2f}, "
    f"Salário={max_row['valor_sal_num']:.2f})"
)

print("\nMENOR CUSTO RELATIVO:")
print(
    f"Data: {min_row['data']:%d/%m/%Y} | "
    f"Edição: {min_row['edicao_numero']} | "
    f"{min_row['%_sal_min']:.4f}% do salário mínimo "
    f"(Gibi={min_row['valor_gibi_num']:.2f}, "
    f"Salário={min_row['valor_sal_num']:.2f})"
)

print("\nESTATÍSTICAS GERAIS:")
print(f"Média: {media:.3f}% do salário mínimo")
print(f"Mediana: {mediana:.3f}% do salário mínimo")